# Continuous $\varepsilon$-Sinkhorn flow

**Credit.** This notebook is part of *Optimal Transport for Machine Learners* by [Gabriel Peyré](https://www.gpeyre.com/), CNRS and École normale supérieure, PSL University. The book and reproducible sources are available at [github.com/gpeyre/ot4ml](https://github.com/gpeyre/ot4ml).

**Copyright.** Copyright (c) 2025 Gabriel Peyré. Released under the MIT License; see the repository `LICENSE` file.

This notebook illustrates the continuous-time potential flow obtained from the high-resolution scaling of Sinkhorn iterations studied by Berman.  One lets the grid resolution $k$ grow while setting the entropic temperature to $\epsilon=k^{-1}$, and observes the scaled log-Sinkhorn iterates at continuous time $t=m/k$.  The vanishing-temperature limit is a parabolic Monge--Ampere equation for a gauge-fixed potential $u_t$,

$$
\partial_t u_t(x)
= \log \det(I+\nabla^2 u_t(x))
  -G(x+\nabla u_t(x))+F(x)-\bar r_t,
$$

where $\alpha=e^{-F}dx$ and $\beta=e^{-G}dx$ are normalized densities on the unit-volume periodic domain.  The scalar $\bar r_t$ fixes the additive gauge of the potential.  In one dimension the determinant becomes $1+u_t''$, and the flow is easy to visualize as curves evolving from the zero initialization.


The two examples below are deliberately smooth.  This keeps the numerical solution inside the convexity regime $1+u''>0$ and makes the picture an illustration of the flow rather than a stress test of a PDE solver.

In [ ]:
# --- OT4ML_COLAB_BOOTSTRAP: make this figure notebook runnable standalone in Colab.
from pathlib import Path as _OT4ML_Path
import importlib.util as _OT4ML_importlib_util
import os as _OT4ML_os
import subprocess as _OT4ML_subprocess
import sys as _OT4ML_sys


def _ot4ml_find_repo_root():
    """Locate the OT4ML repository root for shared OT4ML assets.
    
    Returns:
        Repository root path located by the function.
    """
    here = _OT4ML_Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "notebooks-figures" / "figure_style.py").exists():
            return base
        if base.name == "notebooks-figures" and (base / "figure_style.py").exists():
            return base.parent
    return None


def _ot4ml_ensure_package(module_name, package_name):
    """Install a package when the corresponding module is missing.
    
    Args:
        module_name: Python module name that should be importable.
        package_name: Package name to install when the module is unavailable.
    
    Returns:
        None. Imports the package if available, otherwise installs it first.
    """
    if _OT4ML_importlib_util.find_spec(module_name) is None:
        _OT4ML_subprocess.run(
            [_OT4ML_sys.executable, "-m", "pip", "install", "-q", package_name],
            check=True,
        )


ROOT = _ot4ml_find_repo_root()
if ROOT is None:
    ROOT = _OT4ML_Path("/content/ot4ml") if "google.colab" in _OT4ML_sys.modules else _OT4ML_Path.cwd() / "ot4ml"
    if ROOT.exists() and any(ROOT.iterdir()) and not (ROOT / ".git").exists():
        ROOT = ROOT.with_name(ROOT.name + "-repo")
    if not (ROOT / "notebooks-figures" / "figure_style.py").exists():
        _OT4ML_subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/gpeyre/ot4ml.git", str(ROOT)],
            check=True,
        )

_OT4ML_os.chdir(ROOT)
_figures_path = str(ROOT / "notebooks-figures")
if _figures_path not in _OT4ML_sys.path:
    _OT4ML_sys.path.insert(0, _figures_path)

if "google.colab" in _OT4ML_sys.modules:
    pass

VERBOSE = False

def log(*args, **kwargs):
    """Print diagnostics only when VERBOSE is enabled.
    
    Args:
        *args: Positional arguments forwarded to the wrapped callable.
        **kwargs: Keyword arguments forwarded to the wrapped callable.
    
    Returns:
        None. Prints diagnostic messages only when verbose mode is enabled.
    """
    if VERBOSE:
        print(*args, **kwargs)


In [1]:

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb

repo_root = Path.cwd()
if not (repo_root / "notebooks-figures").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "notebooks-figures"))

from figure_style import BLUE, RED, VIOLET, box_axes, figure_dir, interp_color, save_pdf, setup_matplotlib

setup_matplotlib()

FIGURE_NAME = "sinkhorn-continuous-epsilon-flow"
out_dir = figure_dir(FIGURE_NAME)
thumb_dir = repo_root / "notebooks-figures" / "thumbnails"
thumb_dir.mkdir(parents=True, exist_ok=True)


## Periodic densities and finite differences

We work on the flat torus $\mathbb T=\mathbb R/\mathbb Z$.  The helper below represents densities as exponentials of smooth Fourier potentials, so they are positive and normalized.  Periodic interpolation is used to evaluate $G(x+u')$.

In [2]:

N = 512
x = np.linspace(0.0, 1.0, N, endpoint=False)
dx = 1.0 / N


def normalize_density(rho):
    """Normalize density to a stable probability or display scale.
    
    Args:
        rho: Density or density-like array.
    
    Returns:
        Normalized density to a stable probability or display scale.
    """
    rho = np.maximum(np.asarray(rho, dtype=float), 1e-14)
    rho /= rho.mean()  # integral on the unit torus is one
    return rho


def density_from_fourier(phi):
    """Evaluate a density from Fourier coefficients on the supplied grid.
    
    Args:
        phi: Potential values, phase values, or feature map.
    
    Returns:
        Density from fourier values evaluated on the supplied grid.
    """
    return normalize_density(np.exp(phi))


def periodic_interp(values, position):
    """Compute periodic interp.
    
    Args:
        values: Array of scalar values to integrate, normalize, transform, or display.
        position: Point, label, or panel position.
    
    Returns:
        Periodic interp values.
    """
    position = np.mod(position, 1.0)
    scaled = position * N
    i0 = np.floor(scaled).astype(int) % N
    t = scaled - i0
    return (1.0 - t) * values[i0] + t * values[(i0 + 1) % N]


def centered_log_density(rho):
    """Evaluate centered log density on the supplied grid.
    
    Args:
        rho: Density or density-like array.
    
    Returns:
        Centered log density values evaluated on the supplied grid.
    """
    val = -np.log(normalize_density(rho))
    return val - val.mean()


def smooth_density_cases():
    # A single bump translated to the right.
    """Evaluate smoothed density test cases on the supplied grid.
    
    Returns:
        Smooth density cases values evaluated on the supplied grid.
    """
    alpha_1 = density_from_fourier(0.55 * np.cos(2 * np.pi * (x - 0.25)))
    beta_1 = density_from_fourier(0.55 * np.cos(2 * np.pi * (x - 0.57)))

    # A genuinely different topology of the log-density: one-and-a-half waves toward a bimodal target.
    alpha_2 = density_from_fourier(
        0.35 * np.cos(2 * np.pi * (x - 0.20))
        + 0.16 * np.cos(4 * np.pi * (x - 0.08))
    )
    beta_2 = density_from_fourier(
        0.30 * np.cos(4 * np.pi * (x - 0.17))
        + 0.18 * np.cos(6 * np.pi * (x + 0.02))
    )

    return {
        "unimodal": (alpha_1, beta_1),
        "multimodal": (alpha_2, beta_2),
    }


## Parabolic Monge--Ampere flow

Berman's scaled log-Sinkhorn increment has the form $u_{m+1}^{(k)}-u_m^{(k)}=k^{-1}\log \rho_{k u_m^{(k)}}$.  A Laplace expansion gives $\rho_{ku}(x)=\det(I+\nabla^2u(x))e^{F(x)-G(x+\nabla u(x))}(1+O(k^{-1}))$, which yields the continuous flow below when $m/k\to t$ under Berman's smoothness and strict quasi-convexity assumptions.  The normalized limiting PDE has no explicit $\epsilon$; the word epsilon only appears before the high-resolution rescaling.

The update below is an explicit finite-difference discretization of

$$
\partial_t u_t = \log(1+u_t'') - G(x+u_t') + F(x) - \bar r_t.
$$

The time step is intentionally small.  The code stops if the convexity condition $1+u''>0$ is violated, which would mean that the displayed PDE solution has left the regime where the map $x\mapsto x+u'(x)$ is monotone.


In [3]:

def simulate_pma_flow(alpha, beta, *, dt=5e-7, steps=200_000, n_snapshots=13):
    """Simulate pma flow for the time snapshots shown in the figure.
    
    Args:
        alpha: Source measure, density, or probability weights.
        beta: Target measure, density, or probability weights.
        dt: Time step used by the numerical integration.
        steps: Number of numerical integration steps.
        n_snapshots: Number of objects specified by this parameter.
    
    Returns:
        Recorded particle states or density snapshots at the requested display times.
    """
    F = centered_log_density(alpha)
    G = centered_log_density(beta)
    u = np.zeros_like(x)
    snapshot_steps = np.linspace(0, steps, n_snapshots, dtype=int)
    snapshot_set = set(snapshot_steps.tolist())
    snapshots = []
    diagnostics = []

    for it in range(steps + 1):
        if it in snapshot_set:
            snapshots.append(u.copy())

        ux = (np.roll(u, -1) - np.roll(u, 1)) / (2.0 * dx)
        uxx = (np.roll(u, -1) - 2.0 * u + np.roll(u, 1)) / (dx * dx)
        jac = 1.0 + uxx
        diagnostics.append((jac.min(), jac.max()))
        if jac.min() <= 0:
            raise RuntimeError(f"Convexity condition failed at iteration {it}: min(1+u_xx)={jac.min():.3e}")

        rhs = np.log(jac) - periodic_interp(G, x + ux) + F
        rhs -= rhs.mean()  # gauge fixing

        if it < steps:
            u += dt * rhs
            u -= u.mean()

    return np.asarray(snapshots), np.asarray(diagnostics)


cases = smooth_density_cases()
solutions = {}
for name, (alpha, beta) in cases.items():
    snapshots, diagnostics = simulate_pma_flow(alpha, beta)
    solutions[name] = {
        "alpha": alpha,
        "beta": beta,
        "snapshots": snapshots,
        "diagnostics": diagnostics,
    }
    log(name, "min convexity", diagnostics[:, 0].min(), "final potential range", np.ptp(snapshots[-1]))


unimodal min convexity 0.3931189242775872 final potential range 0.042985071480418705


multimodal min convexity 0.4771232677003354 final potential range 0.021058925804778132


## Plotting

Each exported panel contains only curves.  The bottom silhouettes show the source and target densities with a common vertical rescaling; the potential itself starts from the red zero curve and moves toward blue.  The figure is meant as a clean intuition for the vanishing-temperature PDE limit, not as a benchmark for PDE solvers.


In [4]:

def lighten(color, amount=0.78):
    """Blend a color toward white by a prescribed amount.
    
    Args:
        color: Color used for drawing the object.
        amount: Interpolation amount or perturbation strength.
    
    Returns:
        Lighten values.
    """
    rgb = np.array(to_rgb(color))
    return tuple((1 - amount) * rgb + amount * np.ones(3))


def plot_panel(name, *, summary=False):
    """Plot the panel with the OT4ML plotting style.
    
    Args:
        name: Name used to identify the panel, case, or file.
        summary: Summary statistics or summary text.
    
    Returns:
        None. Draws on the provided Matplotlib axes.
    """
    data = solutions[name]
    snapshots = data["snapshots"]
    alpha = data["alpha"]
    beta = data["beta"]
    times = np.linspace(0.0, 1.0, len(snapshots))

    if summary:
        fig, ax = plt.subplots(figsize=(3.85, 2.25))
    else:
        fig, ax = plt.subplots(figsize=(3.20, 2.35))

    y_min = snapshots.min()
    y_max = snapshots.max()
    span = max(y_max - y_min, 1e-4)
    density_base = y_min - 0.34 * span
    density_height = 0.22 * span
    ax.fill_between(x, density_base, density_base + density_height * alpha / alpha.max(),
                    color=lighten(RED, 0.60), linewidth=0, zorder=0)
    ax.fill_between(x, density_base, density_base + density_height * beta / beta.max(),
                    color=lighten(BLUE, 0.58), linewidth=0, zorder=0)
    ax.plot(x, density_base + density_height * alpha / alpha.max(), color=RED, lw=0.75, alpha=0.65)
    ax.plot(x, density_base + density_height * beta / beta.max(), color=BLUE, lw=0.75, alpha=0.65)

    for t, u in zip(times, snapshots):
        ax.plot(x, u, color=interp_color(float(t)), lw=1.05 if t in (0.0, 1.0) else 0.82, alpha=0.96)

    ax.axhline(0, color="#333333", lw=0.45, alpha=0.35)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(density_base - 0.06 * span, y_max + 0.08 * span)
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_xticklabels(["0", "1/2", "1"])
    ax.set_yticks([])
    ax.set_xlabel(r"$x$", labelpad=1)
    box_axes(ax)
    return fig


for name in solutions:
    fig = plot_panel(name)
    save_pdf(fig, out_dir / f"{name}.pdf", pad_inches=0.045)
    plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.35), constrained_layout=True)
for ax, name in zip(axes, solutions):
    plt.sca(ax)
    # Reuse the plotting code, then move the artists to the contact axis by redrawing inline.
    data = solutions[name]
    snapshots = data["snapshots"]
    alpha = data["alpha"]
    beta = data["beta"]
    times = np.linspace(0.0, 1.0, len(snapshots))
    y_min = snapshots.min()
    y_max = snapshots.max()
    span = max(y_max - y_min, 1e-4)
    density_base = y_min - 0.34 * span
    density_height = 0.22 * span
    ax.fill_between(x, density_base, density_base + density_height * alpha / alpha.max(), color=lighten(RED, 0.60), linewidth=0)
    ax.fill_between(x, density_base, density_base + density_height * beta / beta.max(), color=lighten(BLUE, 0.58), linewidth=0)
    for t, u in zip(times, snapshots):
        ax.plot(x, u, color=interp_color(float(t)), lw=0.92, alpha=0.96)
    ax.axhline(0, color="#333333", lw=0.4, alpha=0.32)
    ax.set_xlim(0, 1)
    ax.set_ylim(density_base - 0.06 * span, y_max + 0.08 * span)
    ax.set_xticks([0, 0.5, 1])
    ax.set_xticklabels(["0", "1/2", "1"])
    ax.set_yticks([])
    ax.set_xlabel(r"$x$", labelpad=1)
    box_axes(ax)

fig.savefig(thumb_dir / f"{FIGURE_NAME}.png", dpi=180, bbox_inches="tight", pad_inches=0.04)
plt.close(fig)


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


The generated files are:

- `OT4ML/figures/sinkhorn-continuous-epsilon-flow/unimodal.pdf`
- `OT4ML/figures/sinkhorn-continuous-epsilon-flow/multimodal.pdf`
- `notebooks-figures/thumbnails/sinkhorn-continuous-epsilon-flow.png`
